# Induced Matching Bootstrap

In this notebook, we consider the information given by subsamples to capture their topological quality using bootstrap of induced matchings.

We recycle code from https://gudhi.inria.fr/python/latest/rips_complex_sklearn_itf_ref.html

Here we compute landscapes of matching diagrams, the matched and unmatched part respectively, and compute their bootstrapped confidence intervals.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os

plots_dir = "figures"
os.makedirs(plots_dir, exist_ok=True)

In [ ]:
import tdqual.topological_data_quality_0 as tdqual

size_sample_X = 20

RandGen = np.random.default_rng(4)
# # Generate Random Sample
Z = tdqual.sampled_circle(0,2,20, RandGen)
Z = np.vstack((Z, tdqual.sampled_circle(0,2,40, RandGen) + [10,0]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,30, RandGen) + [5,10]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,2, RandGen) + [-5,10]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,2, RandGen) + [15,10]))
list_X_Z = []
num_samples = 4
for i in range(num_samples):
    # Take sample of points and reorder
    X_indices = RandGen.choice(Z.shape[0],size_sample_X, replace=False)
    X_indices_compl = list(set(range(Z.shape[0]))-set(X_indices))
    # Reorder Z and get X
    X =Z[X_indices]
    Z_new = np.vstack((X, Z[X_indices_compl]))
    list_X_Z.append((X, Z_new))

In [ ]:
# Plot point cloud
fig, ax = plt.subplots(ncols=1, figsize=(3,3))
X, Z = list_X_Z[0]
ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=2)
ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="x", zorder=1)
ax.set_axis_off()
plt.savefig(plots_dir + "points_0.png")

Let us compute the landscape

In [ ]:
int(X.shape[0]*0.4)

In [ ]:
# Constants for subsample
nb_times = 80
percentage_subsample = 0.3
X, Z = list_X_Z[0]
size_subsample_X = int(X.shape[0]*percentage_subsample)
size_subsample_X = int(X.shape[0])
size_subsample_compl = int((Z.shape[0]-X.shape[0])*percentage_subsample)

def subsample_indices(X, Z, size_subsample_X):
    # Assume that the first X.shape[0] points from Z are those from X
    sub = []
    # construct a list of nb_times x nb_points
    for sub_idx in range(nb_times):
        X_indices = np.random.choice(range(X.shape[0]), size_subsample_X, replace=False)
        compl_indices = np.random.choice(range(X.shape[0]+1, Z.shape[0]), size_subsample_compl, replace=False)
        sub.append(np.vstack((Z[X_indices], Z[compl_indices])))
    return sub

In [ ]:
sub = []
for i in range(len(list_X_Z)):
    sub.append(subsample_indices(list_X_Z[i][0], list_X_Z[i][1], size_subsample_X))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bootstrap
from gudhi.representations import Landscape

# Constants
landscape_resolution = 600

# 1. Initialize the GUDHI Landscape transformer manually
# We set num_landscapes=1 to match your original pipeline configuration
landscape_transformer = Landscape(num_landscapes=1, resolution=landscape_resolution)

def plot_average_landscape(landscapes, color, label, ax):
    """Processes and plots the bootstrapped confidence intervals for the landscapes."""
    rng = np.random.default_rng()
    res = bootstrap(
        (np.transpose(landscapes),),
        np.std,
        method="basic",
        axis=-1,
        confidence_level=0.95,
        random_state=rng,
    )
    ci_l, ci_u = res.confidence_interval
    ax.fill_between(np.arange(0, filter_idx, 1), ci_l, ci_u, alpha=0.3, color=color, label=label)

def process_pairs_landscapes(pairs_subsamples, nb_points_X):
    """
    Replaces the pipeline. Loops through your custom data samples, 
    extracts finite diagram points using your function, and converts them to landscapes.
    """
    finite_landscapes = []
    inf_landscapes = []
    
    for sample in pairs_subsamples:
        X = sample[:nb_points_X]
        Z = sample
        filt_X, filt_Z, matching = filt_X, filt_Z, matching = tdqual.compute_Mf_0(X, Z)
        
        # Call your previous function to split the points
        finite_diag, inf_diag = tdqual.compute_matching_diagram_separated(filt_X, filt_Z, matching)
        
        # GUDHI's Landscape transformer expects a list of diagrams (each a numpy array)
        # If finite_pts is empty, we pass an empty array of shape (0, 2)
        finite_diag = finite_diag if len(finite_diag) > 0 else np.empty((0, 2))
        inf_diag = inf_diag if len(inf_diag) > 0 else np.empty((0, 2))
        finite_landscapes.append(finite_diag)
        inf_landscapes.append(inf_diag)
        
    # Fit/transform the landscapes globally for this activity matrix
    # (In a global setup, you'd want to fit on ALL activities combined first 
    # to lock the internal min/max grid layout, just like pipe.fit did)
    return landscape_transformer.fit_transform(finite_landscapes), landscape_transformer.fit_transform(inf_landscapes)



In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
for i in range(len(sub)):
    finite_landscapes, inf_landscapes = process_pairs_landscapes(sub[i], nb_points_X)
    plot_average_landscape(finite_landscapes, mpl.colormaps["Set1"](i), f"Sub {i}", ax[0])
    plot_average_landscape(inf_landscapes, mpl.colormaps["Set1"](i), f"Sub {i}", ax[1])

ax[0].set_title("Average landscapes (Finite part)")
ax[1].set_title("Average landscapes (Inf part)")
plt.legend()
plt.show()

In [ ]:
# Plot point cloud
fig, ax = plt.subplots(ncols=len(list_X_Z), figsize=(3*len(list_X_Z),3))

for i in range(len(sub)):
    X, Z = list_X_Z[i]
    ax[i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=2)
    ax[i].scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="x", zorder=1)
    #ax[i].set_axis_off()
    ax[i].set_title(f"Set {i}", color = mpl.colormaps["Set1"](i))
plt.show()